In [5]:
# Import Library

import os
import torch
import torchaudio
import librosa
import numpy as np
import soundfile as sf
from pydub import AudioSegment

In [6]:
# Load data + generate const var
bundle = torchaudio.pipelines.WAV2VEC2_ASR_BASE_960H
model = bundle.get_model()

audio_file = "dataset/sample_audio.wav"
audio, sr = librosa.load(audio_file, sr=None)

In [ ]:
# Export function

def export_function(audio):
    while True:
        file_name = input("input filename: ")
        if file_name.endswith(".wav"):
            break
        else:
            print("file name must ends with .wav")
    audio.export(file_name, format="wav")
    print(f"audio saved as {file_name}")


def export_librosa(audio):
    while True:
        file_name = input("input filename: ")
        if file_name.endswith(".wav"):
            break
        else:
            print("file name must ends with .wav")
    sf.write(file_name, np.clip(audio, -1, 1), sr)  # clip: noise/gain can exceed 1.0
    print(f"audio saved as {file_name}")


In [7]:
# Augmentation function
# pydub (clip)   : volume, fade in, fade out, repeat, reverse
# librosa (audio): pitch, white noise, speed, random gain

def adjvolume(clip):
    while True:
        db = float(input("input db (-30 to 30): "))
        if -30 <= db <= 30:
            break
        else:
            print("must be between -30 to 30")

    export_function(clip + db)

def adjpitch():
    while True:
        pitch_rate = float(input("input pitch rate (-5 to 5): "))
        if -5 <= pitch_rate <= 5:
            break
        else:
            print("must be between -5 to 5")

    shifted_audio = librosa.effects.pitch_shift(audio, sr=sr, n_steps=pitch_rate)
    export_librosa(shifted_audio)

def fade_in(clip):
    while True:
        duration = int(input("input duration in ms (1s = 1000ms): "))
        if duration > 0:
            break
        else:
            print("must be more than 0")

    export_function(clip.fade_in(duration))

def fade_out(clip):
    while True:
        duration = int(input("input duration in ms (1s = 1000ms): "))
        if duration > 0:
            break
        else:
            print("must be more than 0")

    export_function(clip.fade_out(duration))

def white_noise():
    while True:
        level = float(input("input noise level (0.1 to 0.5): "))
        if 0.1 <= level <= 0.5:
            break
        else:
            print("must be between 0.1 to 0.5")

    noise = np.random.normal(0, audio.std(), audio.size)
    export_librosa(audio + noise * level)

def repeat_clip(clip):
    while True:
        times = int(input("input repeat count (1 to 100): "))
        if 1 <= times <= 100:
            break
        else:
            print("must be between 1 to 100")

    export_function(clip * times)

def change_speed():
    while True:
        rate = float(input("input speed rate, 0.5 = slower, 2 = faster (0.5 to 2): "))
        if 0.5 <= rate <= 2:
            break
        else:
            print("must be between 0.5 to 2")

    export_librosa(librosa.effects.time_stretch(audio, rate=rate))

def reverse_audio(clip):
    export_function(clip.reverse())

def random_gain():
    gain = np.random.uniform(0.5, 1.5)
    print(f"random gain x{gain:.2f}")

    export_librosa(audio * gain)

In [8]:
# Pred func

def predict_text(audio_file):
    waveform, sr = torchaudio.load(audio_file)
    waveform = waveform.mean(dim=0, keepdim=True)  # stereo -> mono

    if sr != bundle.sample_rate:
        waveform = torchaudio.functional.resample(waveform, sr, bundle.sample_rate)

    with torch.inference_mode():
        emission, _ = model(waveform)

    class GreedyCTCDecoder(torch.nn.Module):
        def __init__(self, labels):
            super().__init__()
            self.labels = labels
            self.blank = 0

        def forward(self, emission: torch.Tensor) -> str:
            indices = torch.argmax(emission, dim=-1)
            indices = torch.unique_consecutive(indices, dim=-1)
            indices = [i for i in indices if i != self.blank]
            return "".join([self.labels[i] for i in indices])

    decoder = GreedyCTCDecoder(bundle.get_labels())
    transcript = decoder(emission[0])
    transcript = transcript.replace("|", " ")
    return transcript

In [9]:
# Main menu augment

def menu1():
    clip = AudioSegment.from_wav(audio_file)
    while True:
        print("1. increase/dec volume")
        print("2. increase/dec pitch")
        print("3. insert fadein")
        print("4. insert fadeout")
        print("5. add white noise")
        print("6. repeat clip")
        print("7. change speed")
        print("8. reverse audio")
        print("9. random gain")
        print("10. back to main menu")

        choice = input(">> ")
        if choice == "1":
            adjvolume(clip)
        elif choice == "2":
            adjpitch()
        elif choice == "3":
            fade_in(clip)
        elif choice == "4":
            fade_out(clip)
        elif choice == "5":
            white_noise()
        elif choice == "6":
            repeat_clip(clip)
        elif choice == "7":
            change_speed()
        elif choice == "8":
            reverse_audio(clip)
        elif choice == "9":
            random_gain()
        elif choice == "10":
            break
        else:
            print("invalid input")

In [ ]:
# Main menu

def mainmenu():
    while True:
        print("1. audio segmentation")
        print("2. predict text")
        print("3. exit")

        choice = int(input(">> "))
        if choice == 1:
            menu1()
        elif choice == 2:
            transcript = predict_text(audio_file)
            print(f"hasil transcript {transcript}")
        elif choice == 3:
            break
        else:
            print("invalid input")
    print("thx")

mainmenu()

1. audio augmentation
2. predict text
3. exit
1. increase/dec volume
2. increase/dec pitch
3. insert fadein
4. insert fadeout
5. add white noise
6. repeat clip
7. change speed
8. reverse audio
9. random gain
10. back to main menu
must be between 0.1 to 0.5
file name must ends with .wav
file name must ends with .wav
audio saved as sample_audio.wav
1. increase/dec volume
2. increase/dec pitch
3. insert fadein
4. insert fadeout
5. add white noise
6. repeat clip
7. change speed
8. reverse audio
9. random gain
10. back to main menu
1. audio augmentation
2. predict text
3. exit
hasil transcript THE PUBLIC ARE ENTREATED TO BEAR IN MIND THAT THIRTEEN YEARS HAVE PASSED SINCE IT WAS FINISHED MANY MORE SINCE IT WAS BEGUN AND THAT DURING THAT PERIOD PLACES MANNERS BOOKS AND OPINIONS HAVE UNDERGONE CONSIDERABLE CHANGES 
1. audio augmentation
2. predict text
3. exit
invalid input
1. audio augmentation
2. predict text
3. exit
thx
